# 01 Dataset Overview

Stage 1 inventory and Stage 2 label-mapping audit for local DREAMT participant CSV files. This notebook checks file presence, expected columns, label and event annotation columns, missingness, approximate recording duration, and sleep-stage target standardization. It intentionally avoids predictive EDA, feature engineering, scaling, modeling, and train/validation/test splitting.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import (
    DEFAULT_INTERIM_DATA_DIR,
    DEFAULT_RAW_DATA_DIR,
    EVENT_ANNOTATION_COLUMNS,
    EXPECTED_DREAMT_COLUMNS,
    EXPECTED_SIGNAL_COLUMNS,
    LABEL_COLUMN,
    list_participant_csvs,
    summarize_dataset,
)

from src.preprocessing import (
    identify_invalid_labels,
    map_sleep_stage,
    summarize_label_mapping,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

raw_data_dir = repo_root / DEFAULT_RAW_DATA_DIR
summary_path = repo_root / DEFAULT_INTERIM_DATA_DIR / "participant_summary.csv"
label_mapping_summary_path = repo_root / DEFAULT_INTERIM_DATA_DIR / "label_mapping_summary.csv"
label_mapping_summary_p_as_wake_path = repo_root / DEFAULT_INTERIM_DATA_DIR / "label_mapping_summary_p_as_wake.csv"
expected_n_participants = 100

## Locate Participant Files

In [ ]:
try:
    participant_files = list_participant_csvs(raw_data_dir)
except FileNotFoundError as exc:
    participant_files = []
    print(exc)

participant_file_table = pd.DataFrame(
    {"file_path": [str(path) for path in participant_files]}
)
print(f"Found {len(participant_files)} participant CSV file(s).")
participant_file_table.head()

## Build Or Load Participant Summary

In [ ]:
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print(f"Loaded existing summary: {summary_path}")
else:
    try:
        summary_df = summarize_dataset(raw_data_dir, output_path=summary_path)
    except FileNotFoundError as exc:
        print(exc)
        summary_df = pd.DataFrame()

summary_df.head()

In [ ]:
def parse_json_cell(value, default=None):
    if default is None:
        default = {}
    if pd.isna(value):
        return default
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return default


def require_summary(summary):
    if summary.empty:
        print("No participant summary is available yet. Add raw CSVs under data/raw/ and rerun the notebook.")
        return False
    return True

## Participant Count And File Integrity

In [ ]:
participant_count_table = pd.DataFrame(
    [
        {
            "expected_participant_files": expected_n_participants,
            "found_participant_files": len(participant_files),
            "all_100_present": len(participant_files) == expected_n_participants,
        }
    ]
)
participant_count_table

In [ ]:
if require_summary(summary_df):
    integrity_columns = [
        "participant_id",
        "n_rows",
        "n_columns",
        "has_expected_schema",
        "has_all_expected_columns",
        "missing_expected_columns",
        "extra_columns",
        "error",
    ]
    display(summary_df[integrity_columns].sort_values(["error", "n_rows"], na_position="last"))

    schema_summary = summary_df["has_expected_schema"].value_counts(dropna=False).rename_axis("has_expected_schema").reset_index(name="n_participants")
    display(schema_summary)

## Missing Signals, Labels, And Event Annotations

In [ ]:
if require_summary(summary_df):
    signal_presence_columns = [f"has_{column}" for column in EXPECTED_SIGNAL_COLUMNS]
    signal_presence = summary_df[["participant_id", *signal_presence_columns]].copy()
    display(signal_presence)

    missing_signal_counts = pd.DataFrame(
        {
            "signal": EXPECTED_SIGNAL_COLUMNS,
            "participants_missing_signal": [
                int((~summary_df[f"has_{column}"].fillna(False).astype(bool)).sum())
                for column in EXPECTED_SIGNAL_COLUMNS
            ],
        }
    )
    display(missing_signal_counts)

In [ ]:
if require_summary(summary_df):
    label_and_event_columns = [
        "participant_id",
        "label_column",
        *[f"has_{column}" for column in EVENT_ANNOTATION_COLUMNS],
    ]
    display(summary_df[label_and_event_columns])

    missing_label_count = int((summary_df["label_column"] != LABEL_COLUMN).sum())
    event_presence_counts = pd.DataFrame(
        {
            "column": EVENT_ANNOTATION_COLUMNS,
            "participants_missing_column": [
                int((~summary_df[f"has_{column}"].fillna(False).astype(bool)).sum())
                for column in EVENT_ANNOTATION_COLUMNS
            ],
        }
    )
    display(pd.DataFrame([{"missing_sleep_stage_participants": missing_label_count}]))
    display(event_presence_counts)

## Empty, Corrupted, Or Unusually Short Files

In [ ]:
if require_summary(summary_df):
    row_threshold = summary_df["n_rows"].quantile(0.05) if summary_df["n_rows"].notna().any() else None
    short_or_problem_files = summary_df[
        summary_df["error"].notna()
        | summary_df["n_rows"].fillna(0).eq(0)
        | (summary_df["n_rows"] < row_threshold if row_threshold is not None else False)
    ][["participant_id", "file_path", "n_rows", "recording_duration_seconds", "error"]]
    display(short_or_problem_files)

    summary_df["n_rows"].dropna().plot(kind="hist", bins=30, title="Rows per participant file")
    plt.xlabel("Rows")
    plt.show()

## Sleep-Stage And Event Annotation Values

In [ ]:
if require_summary(summary_df):
    label_rows = []
    for _, row in summary_df.iterrows():
        for label_value, count in parse_json_cell(row.get("label_counts"), {}).items():
            label_rows.append(
                {
                    "participant_id": row["participant_id"],
                    "sleep_stage_value": label_value,
                    "count": count,
                }
            )
    label_values = pd.DataFrame(label_rows)
    if not label_values.empty:
        display(label_values.groupby("sleep_stage_value", as_index=False)["count"].sum())
    else:
        print("No Sleep_Stage values found in the participant summary.")

In [ ]:
if require_summary(summary_df):
    event_rows = []
    for _, row in summary_df.iterrows():
        event_counts = parse_json_cell(row.get("event_annotation_value_counts"), {})
        for column, counts in event_counts.items():
            for value, count in counts.items():
                event_rows.append(
                    {
                        "participant_id": row["participant_id"],
                        "event_column": column,
                        "event_value": value,
                        "count": count,
                    }
                )
    event_values = pd.DataFrame(event_rows)
    if not event_values.empty:
        display(event_values.groupby(["event_column", "event_value"], as_index=False)["count"].sum())
    else:
        print("No event annotation values found in the participant summary.")

## Missingness By Signal And Participant

In [ ]:
if require_summary(summary_df):
    missingness_rows = []
    for _, row in summary_df.iterrows():
        percentages = parse_json_cell(row.get("missing_value_percentages_by_signal"), {})
        for signal, missing_pct in percentages.items():
            missingness_rows.append(
                {
                    "participant_id": row["participant_id"],
                    "signal": signal,
                    "missing_pct": missing_pct,
                }
            )
    missingness = pd.DataFrame(missingness_rows)
    if not missingness.empty:
        display(missingness.pivot(index="participant_id", columns="signal", values="missing_pct"))
        missingness.groupby("signal")["missing_pct"].mean().sort_values().plot(kind="barh", title="Mean missingness by signal")
        plt.xlabel("Missing values (%)")
        plt.show()
    else:
        print("No signal missingness values found in the participant summary.")

## Approximate Recording Duration

In [ ]:
if require_summary(summary_df):
    duration = summary_df[["participant_id", "recording_duration_seconds"]].copy()
    duration["recording_duration_hours"] = duration["recording_duration_seconds"] / 3600
    display(duration.sort_values("recording_duration_seconds"))

    duration["recording_duration_hours"].dropna().plot(kind="hist", bins=30, title="Recording duration per participant")
    plt.xlabel("Hours")
    plt.show()

## Three-Class Label Mapping

The primary project target is `Wake`, `Non-REM`, and `REM`. PSG stages `N1`, `N2`, and `N3` are grouped as `Non-REM`. DREAMT `data_64Hz` labels include `P` for preparation before PSG recording starts; the primary mapping excludes `P`, while a secondary sensitivity mapping can treat `P` as `Wake` to match the `data_100Hz` convention.


In [ ]:
if participant_files:
    label_mapping_summary = summarize_label_mapping(
        participant_files,
        p_as_wake=False,
        output_path=label_mapping_summary_path,
    )
    label_mapping_summary_p_as_wake = summarize_label_mapping(
        participant_files,
        p_as_wake=True,
        output_path=label_mapping_summary_p_as_wake_path,
    )
    print(f"Saved primary label mapping summary to {label_mapping_summary_path}")
    print(f"Saved P-as-Wake sensitivity summary to {label_mapping_summary_p_as_wake_path}")
else:
    label_mapping_summary = pd.DataFrame()
    label_mapping_summary_p_as_wake = pd.DataFrame()
    print("No participant files found. Add raw CSVs under data/raw/ and rerun this section.")

label_mapping_summary.head()


### Raw And Mapped Label Counts


In [ ]:
if not label_mapping_summary.empty:
    primary_dataset_rows = label_mapping_summary[label_mapping_summary["scope"] == "dataset"]
    raw_counts = primary_dataset_rows[["raw_label", "standardized_label", "mapped_label", "invalid_reason", "count"]]
    display(raw_counts.sort_values(["invalid_reason", "mapped_label", "raw_label"], na_position="last"))

    mapped_counts = (
        primary_dataset_rows.dropna(subset=["mapped_label"])
        .groupby("mapped_label", as_index=False)["count"]
        .sum()
        .sort_values("mapped_label")
    )
    display(mapped_counts)


### Exclusions Under Primary Mapping


In [ ]:
if not label_mapping_summary.empty:
    primary_dataset_rows = label_mapping_summary[label_mapping_summary["scope"] == "dataset"]
    exclusions = (
        primary_dataset_rows.dropna(subset=["invalid_reason"])
        .groupby("invalid_reason", as_index=False)["count"]
        .sum()
        .sort_values("invalid_reason")
    )
    display(exclusions)

    excluded_due_to_p = int(exclusions.loc[exclusions["invalid_reason"] == "Preparation", "count"].sum())
    excluded_missing_or_other = int(exclusions.loc[exclusions["invalid_reason"] != "Preparation", "count"].sum())
    display(pd.DataFrame([
        {
            "excluded_due_to_P": excluded_due_to_p,
            "excluded_due_to_missing_or_other_invalid": excluded_missing_or_other,
        }
    ]))


### Preparation-Stage Counts By Participant


In [ ]:
if not label_mapping_summary.empty:
    participant_p_counts = (
        label_mapping_summary[
            (label_mapping_summary["scope"] == "participant")
            & (label_mapping_summary["standardized_label"] == "P")
        ][["participant_id", "count"]]
        .rename(columns={"count": "p_count"})
        .sort_values(["p_count", "participant_id"], ascending=[False, True])
    )
    display(participant_p_counts)


### Participants With Little Or No Target-Class Coverage


In [ ]:
if not label_mapping_summary.empty:
    participant_target_counts = (
        label_mapping_summary[
            (label_mapping_summary["scope"] == "participant")
            & label_mapping_summary["mapped_label"].notna()
        ]
        .pivot_table(
            index="participant_id",
            columns="mapped_label",
            values="count",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )
    for target_label in ["Wake", "Non-REM", "REM"]:
        if target_label not in participant_target_counts.columns:
            participant_target_counts[target_label] = 0
    class_totals = participant_target_counts[["Wake", "Non-REM", "REM"]].sum(axis=1)
    low_coverage_threshold = 10
    low_or_missing_class_counts = participant_target_counts[
        (participant_target_counts[["Wake", "Non-REM", "REM"]] == 0).any(axis=1)
        | (class_totals < low_coverage_threshold)
    ].sort_values("participant_id")
    display(low_or_missing_class_counts)


### Primary Versus P-As-Wake Sensitivity Counts


In [ ]:
if not label_mapping_summary.empty and not label_mapping_summary_p_as_wake.empty:
    comparison_rows = []
    for mode_name, summary in [
        ("primary_P_excluded", label_mapping_summary),
        ("sensitivity_P_as_Wake", label_mapping_summary_p_as_wake),
    ]:
        dataset_rows = summary[summary["scope"] == "dataset"]
        class_counts = (
            dataset_rows.dropna(subset=["mapped_label"])
            .groupby("mapped_label")["count"]
            .sum()
            .to_dict()
        )
        exclusion_counts = (
            dataset_rows.dropna(subset=["invalid_reason"])
            .groupby("invalid_reason")["count"]
            .sum()
            .to_dict()
        )
        comparison_rows.append({
            "mapping": mode_name,
            "Wake": class_counts.get("Wake", 0),
            "Non-REM": class_counts.get("Non-REM", 0),
            "REM": class_counts.get("REM", 0),
            "excluded_P": exclusion_counts.get("Preparation", 0),
            "excluded_missing_or_other_invalid": sum(
                count for reason, count in exclusion_counts.items() if reason != "Preparation"
            ),
        })
    display(pd.DataFrame(comparison_rows))


## Summary Notes

Use the tables above to document whether all 100 expected participant files are present, whether the known DREAMT schema is consistent, which wearable signals or annotation columns are missing, how raw PSG labels map to the three-class target, and whether any participant files need attention before downstream train-only EDA or modeling begins.
